# Step 02 Keywords (Colab)

This notebook mirrors `step_02_keywords.py`. Mount your Drive, load the `request.json` produced by the pipeline (`data/<usecase>/02_keywords/remote/request.json`), run the extraction on Colab (GPU optional), and finally drop the expected artifacts plus an empty `done.marker` back into the `outputs/` folder.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)


Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
import json
from pathlib import Path, PureWindowsPath

# --- configuration (edit the two lines below if needed) ---
COMPUTER_NAME = "My Laptop"          # name shown under Drive > Othercomputers
PROJECT_SUBDIR = "gnn_project"       # synced repository folder
USECASE = "usecase_alife"       # which usecase you are processing

project_root = Path(f"/content/drive/Othercomputers/{COMPUTER_NAME}/{PROJECT_SUBDIR}")
request_path = project_root / f"data/{USECASE}/02_keywords/remote/request.json"
print("Project root:", project_root)
print("Request path:", request_path)
assert request_path.exists(), "request.json not found ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã¢â‚¬Å“ run the pipeline up to step_02 with colab=true first."

with request_path.open() as f:
    req = json.load(f)

default_win_root = PureWindowsPath(
    r"C:\\Users\\paulb\\Documents\\CyD - nathan's project continuation\\Code work\\workspace\\Paul work\\gnn_project"
)
win_root = PureWindowsPath(req.get("project_root", default_win_root))

def win_to_colab(path_str: str) -> Path:
    rel = PureWindowsPath(path_str).relative_to(win_root)
    return project_root.joinpath(*rel.parts)

inputs = win_to_colab(req["inputs_dir"])
outputs = win_to_colab(req["outputs_dir"])
params = req["params"]

outputs.mkdir(parents=True, exist_ok=True)
print("Inputs:", inputs)
print("Outputs:", outputs)
print("Params loaded.")
print("sample_papers =", params["sample_papers"])
print("request file  =", request_path)

Project root: /content/drive/Othercomputers/My Laptop/gnn_project
Request path: /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_alife/02_keywords/remote/request.json
Inputs: /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_alife/01_extract/outputs
Outputs: /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_alife/02_keywords/outputs
Params loaded.
sample_papers = 3000
request file  = /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_alife/02_keywords/remote/request.json


In [ ]:
!pip install --quiet keybert sentence-transformers scikit-learn

In [ ]:
import sys
from typing import List

import pandas as pd
from tqdm import tqdm

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

from steps.step_02_core import (
    load_abstracts,
    ensure_cols,
    normalize_text,
    clean_text,
    normalize_for_span_check,
    has_only_whitespace_separators,
    token_set,
    jaccard,
    make_extractor,
    extract_keywords_batch,
    build_keyword_tables,
    DEFAULT_MODEL,
    DEFAULT_TOP_N,
    DEFAULT_RANGE,
    DEFAULT_NR_CANDIDATES,
    DEFAULT_DIVERSITY,
    DEFAULT_STOP_WORDS,
    DEFAULT_STRICT_LITERAL,
    DEFAULT_JACCARD_THRESHOLD,
)



In [ ]:
import os

os.chdir(project_root)

kind, extractor = make_extractor(params)
ngram_values = params.get('keyword_ngram', list(DEFAULT_RANGE))
if isinstance(ngram_values, (list, tuple)) and len(ngram_values) >= 2:
    ngram = (int(min(ngram_values)), int(max(ngram_values)))
else:
    ngram = tuple(DEFAULT_RANGE)
topn = int(params.get('keyword_topn', DEFAULT_TOP_N))
nr_candidates = int(params.get('keyword_nr_candidates', DEFAULT_NR_CANDIDATES))
diversity = float(params.get('keyword_diversity', DEFAULT_DIVERSITY))
stop_words = params.get('keyword_stop_words', DEFAULT_STOP_WORDS)
strict_literal = bool(params.get('keyword_strict_literal', DEFAULT_STRICT_LITERAL))
jaccard_threshold = float(params.get('keyword_jaccard_threshold', DEFAULT_JACCARD_THRESHOLD))

df_raw, src_path = load_abstracts(inputs, params['abstracts_candidates'])
df = ensure_cols(df_raw, params.get('abstracts_text_col', 'Abstract'), params['paper_id_candidates'])

limit = params.get('sample_papers')
n_papers = len(df)

if limit:
    if n_papers > limit:
        df = df.sample(n=int(limit), random_state=42).reset_index(drop=True)
        print(f"[step_02] Sampled {limit} abstracts out of {n_papers}.")
    else:
        print(f"[step_02] Only {n_papers} abstracts available (< {limit}); sampling skipped.")

df.head()


[step_02] Sampled 3000 abstracts out of 49512.


,paper_id,abstract
0,https://openalex.org/W1003506,in this paper a new cellular automata model su...
1,https://openalex.org/W4392222412,for approximate inference in the generalized q...
2,https://openalex.org/W1554794238,current approaches to characterize the complex...
3,https://openalex.org/W1563286165,valiant s shift problem asks whether all n cyc...
4,https://openalex.org/W3014527550,in modern systems data security is needed more...


In [ ]:
abstracts = df['abstract'].tolist()
threads = int(params.get('threads', 1))
extraction = extract_keywords_batch(
    abstracts,
    params,
    extractor=extractor,
    kind=kind,
    progress_desc='[Colab] extracting',
    progress_enabled=True,
    threads=threads,
    return_per_doc=True,
)
keywords_per_doc = extraction.keywords_per_doc or [[] for _ in range(len(abstracts))]
flat_keywords = extraction.flat_keywords

df['keywords'] = keywords_per_doc
(outputs / 'keywords_from_abstract.csv').write_text(
    df[['paper_id', 'keywords']].to_csv(index=False), encoding='utf-8'
)


[Colab] extracting: 100%|██████████| 3000/3000 [05:08<00:00,  9.72doc/s]


1044330

In [ ]:
tables = build_keyword_tables(flat_keywords, params)

tables.df_keys[["Keyword", "Keyword_norm", "Count"]].to_csv(
    outputs / "cleaned_keywords_to_build_graphs.csv", index=False
)
pd.DataFrame({"Keyword": tables.cleaned_list}).to_csv(
    outputs / "cleaned_keyword_list.csv", index=False
)
tables.df_keys.head()


,Keyword_norm,Count,Keyword
0,cellular automata,76,cellular automata
1,cellular automata ca,59,cellular automata ca
2,cellular neural networks,31,cellular neural networks
3,finite type,29,finite type
4,random number generator,28,random number generator


In [ ]:
if not tables.df_hits.empty:
    tables.df_hits.to_csv(outputs / "keyword_presence_matches.csv", index=False)
elif (outputs / "keyword_presence_matches.csv").exists():
    (outputs / "keyword_presence_matches.csv").unlink()

tables.presence_summary.to_csv(outputs / "keyword_presence_summary.csv", index=False)

tables.df_keys[["Keyword", "Keyword_norm", "Count"]].to_parquet(
    outputs / "keywords.parquet", index=False
)

pd.DataFrame({
    "n_papers": [int(len(df))],
    "n_keywords_raw": [int(len(flat_keywords))],
    "n_keywords_unique": [int(tables.df_keys.shape[0])],
    "topn": [int(params.get('keyword_topn', DEFAULT_TOP_N))],
    "source_file": [str(src_path)]
}).to_csv(outputs / "keywords_summary.csv", index=False)

(outputs / "done.marker").write_text("ok", encoding="utf-8")
print("Artifacts written to:", outputs)
!ls "$outputs"


Artifacts written to: /content/drive/Othercomputers/My Laptop/gnn_project/data/usecase_alife/02_keywords/outputs
cleaned_keyword_list.csv	      keywords_from_abstract.csv
cleaned_keywords_to_build_graphs.csv  keywords.parquet
done.marker			      keywords_summary.csv
keyword_presence_matches.csv	      used_config.json
keyword_presence_summary.csv


Once Drive sync finishes, rerun the pipeline locally (without `--force`) so stepÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¯02 sees `done.marker` and the new files, then continue with stepÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¯03.